# 01g -- dim_division Seed

**Purpose**: Season-scoped map of stable **conference** (`A`/`B`) to the
season's themed **division name** (`Riddell`/`Wilson` for 2026-2027). Lets a
consumer resolve `dim_fantasy_teams.conference` to a label by
`(season_id, conference)` join instead of a downstream `IF conference="A"`
conditional (ADR-0005, `dim_position`/`dim_school` transformer pattern).

**Key**: `(season_id, conference)` (composite PK).

**Why a separate dim**: `dim_fantasy_teams.division` carries only the *current*
season's point-in-time name (ingested from the Sheet by 01c). `dim_division`
holds the season-scoped history, so a future rename does not erase the prior
season's label. Names are **derived** from `dim_fantasy_teams` (not hardcoded)
so the Sheet stays the single source of the themed names.

**Read-side only**: this is the buildable half of ADR-0005. The Sheet
**write**-sync stays gated (Sheets-API auth + PII go-ahead).

**Horizon**: only seasons whose division names are known. v1 seeds the current
season (2026-2027); append future seasons here as they gain themed names.

**Output**: `data/dim_division.parquet`

In [ ]:
# ---- Setup & Config (shared module) ----------------------------------------
import sys
from pathlib import Path
for _p in (Path.cwd() / "notebooks", Path.cwd(), Path.cwd().parent):
    if (_p / "etl_helpers.py").exists():
        sys.path.insert(0, str(_p)); break
import etl_helpers as etl
from etl_helpers import CFG, DATA

import pandas as pd

## 1 -- Build

In [ ]:
# ---- Build dim_division ----------------------------------------------------
# Current league season (matches the dim_season convention f"{sy}-{ey}").
START_YEAR = CFG.draft_year            # 2026
SEASON_ID = f"{START_YEAR}-{START_YEAR + 1}"

# Derive the conference -> division-name map from the live team manifest (the
# Sheet's Division column, ingested by 01c). Single source for the themed names.
teams = pd.read_parquet(DATA / "dim_fantasy_teams.parquet")
cmap = (teams[["conference", "division"]]
        .dropna()
        .drop_duplicates()
        .sort_values("conference")
        .reset_index(drop=True))

# Guard: each conference must resolve to exactly one division name.
assert set(cmap["conference"]) == set("AB"[:CFG.num_conferences]), (
    f"expected conferences {sorted(set('AB'[:CFG.num_conferences]))}; "
    f"got {sorted(cmap['conference'])}")
assert cmap["conference"].is_unique, (
    "each conference must map to exactly one division name in dim_fantasy_teams")

dim_division = pd.DataFrame({
    "season_id": SEASON_ID,
    "conference": cmap["conference"],
    "division_name": cmap["division"],
})

out_path = DATA / "dim_division.parquet"
dim_division.to_parquet(out_path, index=False)
print(f"dim_division: {len(dim_division)} rows -> {out_path}")
dim_division

## 2 -- Validation

In [ ]:
# ---- Validation ------------------------------------------------------------
df = pd.read_parquet(DATA / "dim_division.parquet")
assert df.set_index(["season_id", "conference"]).index.is_unique, (
    "(season_id, conference) must be unique (composite PK)")
assert set(df["conference"]) == set("AB"[:CFG.num_conferences])
assert df["division_name"].notna().all(), "division_name must be non-null"
print("dim_division OK")
print(df.to_string(index=False))